### the debug by langsmith and the langgraph studio



cloude how tod debug

## anotaion
testing 

paltform etc rerfere the langgraph doc

In [1]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import Annotated, TypedDict
# anntatition is reducer when i create a data get and store one by one inise the list messgaes
from langgraph.graph.message import add_messages


In [7]:
class State(TypedDict):
    messages: Annotated[list[dict], add_messages]

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"]=API_KEY
Travelly=os.getenv("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"]=Travelly
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
llm = ChatGroq(model="qwen/qwen3-32b", api_key=API_KEY)

In [3]:
from langchain_tavily import TavilySearch
tool_travely=TavilySearch(max_results=3)


In [4]:
def multipal(a, b):
    """
    This function takes two arguments, a and b, and returns their product.
    
    Parameters:
    a (int): The first integer.
    b (int): The second integer.
    
    Returns:
    int: The product of a and b
    """
    return a * b

In [ ]:
# tools=[tool_travely,multipal]
# llm_with_tool = llm.bind_tools(tools)
# llm_with_tool.invoke("What is the capital of India?")


AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the capital of India. I need to provide an accurate answer. Let me think about the tools available. There\'s the Tavily Search function which can retrieve up-to-date information. Since the capital of a country is a factual piece of information, I can use Tavily to confirm the current capital. New Delhi has been the capital of India since 1911, but I should check if there have been any recent changes. The Tavily Search function can help verify this. I\'ll use it with the query "current capital of India" to get the most recent and reliable information. I don\'t need to use any other tools since this is a straightforward question. Let me set up the search parameters. The query is "current capital of India". No need to specify domains or date ranges because it\'s a general fact. The search depth can be basic. Alright, I\'ll call Tavily Search with these parameters.\n', 'tool_calls': [{'id': '79jthyd

In [8]:
def tool_calling_llm_if_answer_is_extrenal(state: State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

In [ ]:
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
 ## USED TO TRACKING THE THINGS

In [ ]:

from mcp.server.fastmcp import FastMCP


# tool name
mcp=FastMCP("wheather")
@mcp.tool()  # ✅ Correct decorator usage
async def weather(city: str) -> str:
    """Get weather of the city"""
    return tool_travely.invoke(f"what is the weather of {city}?")


tools = [tool_travely, weather]
from langgraph.prebuilt import ToolNode


graph =StateGraph(State)
graph.add_node("llm_node",tool_calling_llm_if_answer_is_extrenal)
graph.add_node("whether",whether)
graph.add_node("tooling", ToolNode(tools))

#    ex: try to run ythe stdio in cmd look changes in classmethodcode m_whether.py
 # 
from langgraph.prebuilt import tools_condition

graph.add_edge(START, "llm_node")

graph.add_conditional_edges(
    "llm_nodes",
    tools_condition,   # Using the imported tools_condition
    {
        True: "tooling",
        True: "whether",
        False: END
    }
)
graph.add_edge("whether", "llm_node")
graph.add_edge("tooling", "llm_node")



In [22]:
graph=graph.compile()

ValueError: Found edge starting at unknown node 'llm_nodes'

In [ ]:
from IPython.display import display, Markdown, Image
builder=graph.compile()
try:
        img_bytes = graph.get_graph().draw_mermaid_png(max_retries=3, retry_delay=1.0)
        display(Image(img_bytes))
except Exception:
        # fallback to showing raw mermaid code in case rendering isn't available
        mermaid_code = graph.get_mermaid()
        display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

ValueError: Found edge starting at unknown node 'llm_nodes'